# Generate netcdfs for ECCOv5 llc270

In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import timedelta, datetime
import glob
import os
import re

## import ECCO utils
import sys
sys.path.append('/Users/mzahn/github_others/ECCOv4-py')
import ecco_v4_py as ecco

In [14]:
def load_eccov5_year_by_index(
    year,
    geom,
    var_name="THETA",
    data_dir=None,
    output_dir=None,
):
    
    start_year = 1992
    end_year = 2024

    if year < start_year or year > end_year:
        raise ValueError(f"Year must be between {start_year} and {end_year}")

    # --- Set default directories based on variable name
    if data_dir is None:
        data_dir = f"/home/jpluser/efs-mount-point/mzahn/data/ecco_v5/{var_name}_mon_mean"

    if output_dir is None:
        output_dir = f"/home/jpluser/efs-mount-point/mzahn/data/ecco_v5_nc/{var_name}_mon_mean"

    data_dir = Path(data_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # --- Get full sorted file list
    all_files = sorted(data_dir.glob("*.data"))
    n_expected = (end_year - start_year + 1) * 12

    if len(all_files) != n_expected:
        print(f"Warning: Expected {n_expected} files but found {len(all_files)}")

    # --- Compute slice indices
    year_index = year - start_year
    start_idx = year_index * 12
    end_idx = start_idx + 12

    year_files = all_files[start_idx:end_idx]

    if len(year_files) != 12:
        print(f"Warning: Found {len(year_files)} files for {year}")

    # --- Create time coordinate
    time_coord = (
        pd.date_range(f"{year}-01-01", f"{year}-12-01", freq="MS")
        + pd.Timedelta(days=14)
    )

    # --- Specify c-grid geometries for dataset
    # --- open metadata csv file (Have it in the same directory)
    variable_metadata = pd.read_csv('git_repos/Arctic_heat_and_ice/llc270/eccov5_llc270_variable_metadata.csv')

    # --- Read data and create xarray DataArrays
    arrays = []

    for i, filepath in enumerate(year_files):
        if var_name in ("TFLUX", "oceQsw"):
            array = ecco.read_llc_to_tiles(
                str(filepath.parent),
                filepath.name,
                llc=270,
                nk=1,
                less_output=True,
            )
        else:
            array = ecco.read_llc_to_tiles(
                str(filepath.parent),
                filepath.name,
                llc=270,
                nk=50,
                less_output=True,
            )
            
        ds = ecco.llc_tiles_to_xda(
            array,
            var_type="c",
            dim4="depth",
        )

        # ------------------------------------------------------
        # Rename variable to user-specified name
        # ------------------------------------------------------
        ds = ds.rename(var_name)

        ds = ds.expand_dims("time")
        ds = ds.assign_coords(time=[time_coord[i]])

        arrays.append(ds)

    combined = xr.concat(arrays, dim="time")

    # ----------------------------------------------------------
    # Clear coordinate attributes and copy from geometry
    # ----------------------------------------------------------
    for coord_name in combined.coords:
        combined[coord_name].attrs = {}
        if coord_name in geom.dims:
            combined[coord_name].attrs = geom[coord_name].attrs.copy()
            
    # remove unwanted coords
    # combined = combined.drop_vars(["PHrefC", "Z", "drF"])

    # --- Save NetCDF with variable name in filename
    out_file = output_dir / f"{var_name}_mon_mean_{year}.nc"
    print(out_file)

    combined.to_netcdf(out_file)

    print(f"Saved {out_file}")

    return combined

In [15]:
geom = xr.open_dataset('/home/jpluser/efs-mount-point/mzahn/data/ecco_v5/grid/ECCO-GRID.nc')

# loop through all vars
var_names = ["TFLUX", "oceQsw", "DFrI_TH"]
# var_names = ["ADVr_TH", "ADVy_TH", "ADVx_TH", "TFLUX", "oceQsw", "DFrI_TH", "DFyE_TH", "DFrE_TH", "DFxE_TH"]

for var in var_names:
    print(f"\n================ Processing {var} ================\n")
    for year in range(1992, 2025):
        load_eccov5_year_by_index(year, geom, var_name=var)

================ Processing TFLUX ================

/home/jpluser/efs-mount-point/mzahn/data/ecco_v5_nc/TFLUX_mon_mean/TFLUX_mon_mean_1992.nc
Saved /home/jpluser/efs-mount-point/mzahn/data/ecco_v5_nc/TFLUX_mon_mean/TFLUX_mon_mean_1992.nc
/home/jpluser/efs-mount-point/mzahn/data/ecco_v5_nc/TFLUX_mon_mean/TFLUX_mon_mean_1993.nc
Saved /home/jpluser/efs-mount-point/mzahn/data/ecco_v5_nc/TFLUX_mon_mean/TFLUX_mon_mean_1993.nc
/home/jpluser/efs-mount-point/mzahn/data/ecco_v5_nc/TFLUX_mon_mean/TFLUX_mon_mean_1994.nc
Saved /home/jpluser/efs-mount-point/mzahn/data/ecco_v5_nc/TFLUX_mon_mean/TFLUX_mon_mean_1994.nc
/home/jpluser/efs-mount-point/mzahn/data/ecco_v5_nc/TFLUX_mon_mean/TFLUX_mon_mean_1995.nc
Saved /home/jpluser/efs-mount-point/mzahn/data/ecco_v5_nc/TFLUX_mon_mean/TFLUX_mon_mean_1995.nc
/home/jpluser/efs-mount-point/mzahn/data/ecco_v5_nc/TFLUX_mon_mean/TFLUX_mon_mean_1996.nc
Saved /home/jpluser/efs-mount-point/mzahn/data/ecco_v5_nc/TFLUX_mon_mean/TFLUX_mon_mean_1996.nc
/home/jpluser/efs-

In [9]:
# open one file
tmp = xr.open_dataset('/home/jpluser/efs-mount-point/mzahn/data/ecco_v5_nc/THETA_mon_mean/THETA_mon_mean_2020.nc')

In [10]:
tmp

<xarray.Dataset> Size: 2GB
Dimensions:  (k: 50, tile: 13, j: 270, i: 270, time: 12)
Coordinates:
  * k        (k) int64 400B 0 1 2 3 4 5 6 7 8 9 ... 41 42 43 44 45 46 47 48 49
  * tile     (tile) int64 104B 0 1 2 3 4 5 6 7 8 9 10 11 12
  * j        (j) int64 2kB 0 1 2 3 4 5 6 7 8 ... 262 263 264 265 266 267 268 269
  * i        (i) int64 2kB 0 1 2 3 4 5 6 7 8 ... 262 263 264 265 266 267 268 269
  * time     (time) datetime64[ns] 96B 2020-01-15 2020-02-15 ... 2020-12-15
Data variables:
    THETA    (time, k, tile, j, i) float32 2GB ...